In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score



In [2]:
df = pd.read_csv("anemia_dataset.csv")  


In [3]:
X = df.drop("Anemia", axis=1)
y = df["Anemia"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer


In [6]:
numeric_features = [col for col in X.columns if col != "Gender"]
categorical_features = ["Gender"]

# Preprocessing with ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OrdinalEncoder(), categorical_features)
])

In [7]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("selector", SelectKBest(score_func=f_classif)),
    ("model", XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

In [12]:
param_grid = {
    "selector__k": ["all"],
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.1],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

In [13]:
grid.fit(X_train, y_train)
best_model = grid.best_estimator_

best_model.background_data = X_train.sample(n=100, random_state=42)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


c:\Users\fatim\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [01:08:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [14]:
# Predict on test data directly 
y_pred = best_model.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      0.96      0.93      1200
           1       0.96      0.88      0.92      1200

    accuracy                           0.92      2400
   macro avg       0.93      0.92      0.92      2400
weighted avg       0.93      0.92      0.92      2400



In [15]:
import joblib
joblib.dump(best_model, "xgb_model3.joblib")

['xgb_model3.joblib']